In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Live_Query_Engine_Test") \
    .master("local[2]") \
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1,"
            "org.apache.hadoop:hadoop-aws:3.4.1,"
            "software.amazon.awssdk:bundle:2.29.38") \
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local",           "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type",      "rest") \
    .config("spark.sql.catalog.local.uri",       "http://rest-catalog:8181") \
    .config("spark.sql.catalog.local.warehouse", "s3a://business-data/") \
    .config("spark.sql.catalog.local.s3.endpoint",          "http://minio:9000") \
    .config("spark.sql.catalog.local.s3.path-style-access", "true") \
    .config("spark.sql.catalog.local.s3.region",            "us-east-1") \
    .config("spark.hadoop.fs.s3a.endpoint",          "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key",        "admin") \
    .config("spark.hadoop.fs.s3a.secret.key",        "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print(f"Spark Session Ready! Version: {spark.version}")

Spark Session Ready! Version: 4.0.2


In [2]:
# Query the live Iceberg table
spark.sql("""
    SELECT 
        traffic_event_ts, 
        traffic_borough, 
        traffic_speed, 
        aq_pm25_ugm3, 
        weather_temperature
    FROM local.db.enriched_traffic
    ORDER BY traffic_event_ts DESC
    LIMIT 10
""").show(truncate=False)

+-------------------+---------------+-------------+------------+-------------------+
|traffic_event_ts   |traffic_borough|traffic_speed|aq_pm25_ugm3|weather_temperature|
+-------------------+---------------+-------------+------------+-------------------+
|2026-05-06 17:48:12|Queens         |13.67        |14.8        |64                 |
|2026-05-06 17:48:12|Queens         |13.67        |14.8        |65                 |
|2026-05-06 17:48:12|Queens         |13.67        |14.8        |63                 |
|2026-05-06 17:48:12|Queens         |13.67        |14.8        |59                 |
|2026-05-06 17:48:12|Queens         |13.67        |14.8        |62                 |
|2026-05-06 17:48:12|Queens         |13.67        |14.8        |65                 |
|2026-05-06 17:48:12|Queens         |13.67        |14.8        |64                 |
|2026-05-06 17:48:12|Queens         |13.67        |14.8        |63                 |
|2026-05-06 17:48:12|Queens         |13.67        |14.8        |5

In [3]:
# Verify the aggregations and non-null air quality readings
spark.sql("""
    SELECT 
        traffic_borough, 
        COUNT(*) as total_records,
        ROUND(AVG(traffic_speed), 2) as avg_speed_mph, 
        ROUND(AVG(aq_pm25_ugm3), 2) as avg_pm25,
        MAX(weather_temperature) as current_temp
    FROM local.db.enriched_traffic
    GROUP BY traffic_borough
    ORDER BY avg_speed_mph ASC
""").show(truncate=False)

+---------------+-------------+-------------+--------+------------+
|traffic_borough|total_records|avg_speed_mph|avg_pm25|current_temp|
+---------------+-------------+-------------+--------+------------+
|Manhattan      |1316         |12.42        |6.92    |65          |
|Bronx          |594          |18.43        |6.51    |65          |
|Brooklyn       |321          |19.28        |7.96    |65          |
|Staten Island  |406          |23.94        |9.05    |65          |
|Queens         |677          |30.92        |11.29   |65          |
+---------------+-------------+-------------+--------+------------+



In [4]:
# 1. Read the raw Parquet dumps directly from MinIO
raw_traffic = spark.read.parquet("s3a://raw-data/traffic/")
raw_traffic.createOrReplaceTempView("raw_traffic_view")

# 2. See how many records have successfully landed
spark.sql("""
    SELECT borough, COUNT(*) as rows_ingested 
    FROM raw_traffic_view 
    GROUP BY borough
""").show()

+-------------+-------------+
|      borough|rows_ingested|
+-------------+-------------+
|       Queens|          962|
|     Brooklyn|          312|
|Staten Island|          676|
|    Manhattan|          676|
|        Bronx|          624|
+-------------+-------------+



In [5]:
# Borough-level diagnostics: traffic-only vs traffic+AQ vs traffic+AQ+weather
spark.sql("""
WITH traffic_only AS (
  SELECT borough AS traffic_borough, COUNT(*) AS traffic_rows
  FROM parquet.`s3a://raw-data/traffic/`
  WHERE borough IS NOT NULL
  GROUP BY borough
),
aq_joined AS (
  SELECT traffic_borough, COUNT(*) AS traffic_aq_rows
  FROM local.db.enriched_traffic
  GROUP BY traffic_borough
),
aq_weather_joined AS (
  SELECT traffic_borough, COUNT(*) AS traffic_aq_weather_rows
  FROM local.db.enriched_traffic
  WHERE weather_temperature IS NOT NULL
  GROUP BY traffic_borough
)
SELECT
  t.traffic_borough,
  t.traffic_rows,
  COALESCE(a.traffic_aq_rows, 0) AS traffic_aq_rows,
  COALESCE(w.traffic_aq_weather_rows, 0) AS traffic_aq_weather_rows,
  ROUND(100.0 * COALESCE(a.traffic_aq_rows, 0) / t.traffic_rows, 2) AS aq_match_pct,
  ROUND(100.0 * COALESCE(w.traffic_aq_weather_rows, 0) / t.traffic_rows, 2) AS aq_weather_match_pct
FROM traffic_only t
LEFT JOIN aq_joined a ON t.traffic_borough = a.traffic_borough
LEFT JOIN aq_weather_joined w ON t.traffic_borough = w.traffic_borough
ORDER BY t.traffic_rows DESC
""").show(truncate=False)

+---------------+------------+---------------+-----------------------+------------+--------------------+
|traffic_borough|traffic_rows|traffic_aq_rows|traffic_aq_weather_rows|aq_match_pct|aq_weather_match_pct|
+---------------+------------+---------------+-----------------------+------------+--------------------+
|Queens         |976         |677            |270                    |69.36       |27.66               |
|Staten Island  |688         |406            |120                    |59.01       |17.44               |
|Manhattan      |686         |1316           |1030                   |191.84      |150.15              |
|Bronx          |625         |594            |330                    |95.04       |52.80               |
|Brooklyn       |312         |321            |200                    |102.88      |64.10               |
+---------------+------------+---------------+-----------------------+------------+--------------------+



In [6]:
spark.read.parquet("s3a://raw-data/weather/").selectExpr(
    "MIN(kafka_timestamp) as oldest", "MAX(kafka_timestamp) as newest"
).show(truncate=False)

spark.read.parquet("s3a://raw-data/openaq/").selectExpr(
    "MIN(kafka_timestamp) as oldest", "MAX(kafka_timestamp) as newest"
).show(truncate=False)

+-----------------------+-----------------------+
|oldest                 |newest                 |
+-----------------------+-----------------------+
|2026-05-06 18:01:03.829|2026-05-06 19:07:50.669|
+-----------------------+-----------------------+

+-----------------------+-----------------------+
|oldest                 |newest                 |
+-----------------------+-----------------------+
|2026-05-06 18:01:04.586|2026-05-06 19:09:04.184|
+-----------------------+-----------------------+



In [7]:
spark.sql("""
    SELECT MIN(traffic_event_ts), MAX(traffic_event_ts), COUNT(*) as total
    FROM local.db.enriched_traffic
""").show(truncate=False)

+---------------------+---------------------+-----+
|min(traffic_event_ts)|max(traffic_event_ts)|total|
+---------------------+---------------------+-----+
|2026-05-06 16:03:00  |2026-05-06 17:48:12  |3314 |
+---------------------+---------------------+-----+



In [8]:
spark.sql("""
    SELECT 
        MAX(traffic_event_ts) as latest_traffic,
        (SELECT MIN(kafka_timestamp) FROM parquet.`s3a://raw-data/openaq/`) as earliest_aq
    FROM local.db.enriched_traffic
""").show(truncate=False)

+-------------------+-----------------------+
|latest_traffic     |earliest_aq            |
+-------------------+-----------------------+
|2026-05-06 17:48:12|2026-05-06 18:01:04.586|
+-------------------+-----------------------+



In [9]:
spark.sql("""
    SELECT 
        traffic_borough,
        traffic_id,
        COUNT(*) as aq_matches_per_traffic_row
    FROM local.db.enriched_traffic
    GROUP BY traffic_borough, traffic_id
    ORDER BY aq_matches_per_traffic_row DESC
    LIMIT 20
""").show(truncate=False)

+---------------+----------+--------------------------+
|traffic_borough|traffic_id|aq_matches_per_traffic_row|
+---------------+----------+--------------------------+
|Manhattan      |445       |131                       |
|Manhattan      |221       |111                       |
|Manhattan      |215       |111                       |
|Manhattan      |223       |91                        |
|Manhattan      |217       |91                        |
|Manhattan      |145       |91                        |
|Manhattan      |150       |91                        |
|Brooklyn       |154       |71                        |
|Manhattan      |149       |71                        |
|Brooklyn       |148       |71                        |
|Brooklyn       |157       |71                        |
|Manhattan      |448       |71                        |
|Manhattan      |106       |71                        |
|Manhattan      |124       |71                        |
|Manhattan      |364       |61                  

In [10]:
spark.sql("""
    SELECT 
        h3_index,
        COUNT(*) as rows
    FROM local.db.enriched_traffic
    GROUP BY h3_index
    ORDER BY rows DESC
    LIMIT 10
""").show(truncate=False)

+---------------+----+
|h3_index       |rows|
+---------------+----+
|872a100aeffffff|369 |
|872a1072dffffff|364 |
|872a10729ffffff|355 |
|872a100d2ffffff|283 |
|872a10623ffffff|186 |
|872a100f2ffffff|155 |
|872a1008bffffff|142 |
|872a1072cffffff|131 |
|872a1000bffffff|94  |
|872a1001effffff|84  |
+---------------+----+



In [11]:
spark.sql("""
SELECT
  MIN(traffic_event_ts) AS min_traffic,
  MAX(traffic_event_ts) AS max_traffic,
  MIN(aq_event_ts) AS min_aq,
  MAX(aq_event_ts) AS max_aq,
  MIN(weather_event_ts) AS min_weather,
  MAX(weather_event_ts) AS max_weather
FROM local.db.enriched_traffic
""").show(truncate=False)

spark.sql("""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) AS aq_rows,
  SUM(CASE WHEN weather_temperature IS NOT NULL THEN 1 ELSE 0 END) AS weather_rows
FROM local.db.enriched_traffic
""").show(truncate=False)

+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+
|min_traffic        |max_traffic        |min_aq             |max_aq             |min_weather        |max_weather        |
+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+
|2026-05-06 16:03:00|2026-05-06 17:48:12|2026-05-06 18:01:03|2026-05-06 18:03:05|2026-05-06 17:53:28|2026-05-06 18:01:07|
+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+

+----------+-------+------------+
|total_rows|aq_rows|weather_rows|
+----------+-------+------------+
|3314      |1950   |1950        |
+----------+-------+------------+



In [12]:
# AQ Debugging

spark.sql("""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) AS aq_matched_rows,
  ROUND(100.0 * SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS aq_match_pct
FROM local.db.enriched_traffic
""").show(truncate=False)

+----------+---------------+------------+
|total_rows|aq_matched_rows|aq_match_pct|
+----------+---------------+------------+
|3314      |1950           |58.84       |
+----------+---------------+------------+



In [13]:
spark.sql("""
SELECT
  MIN(traffic_event_ts) AS min_traffic, MAX(traffic_event_ts) AS max_traffic,
  MIN(aq_event_ts) AS min_aq, MAX(aq_event_ts) AS max_aq
FROM local.db.enriched_traffic
""").show(truncate=False)

+-------------------+-------------------+-------------------+-------------------+
|min_traffic        |max_traffic        |min_aq             |max_aq             |
+-------------------+-------------------+-------------------+-------------------+
|2026-05-06 16:03:00|2026-05-06 17:48:12|2026-05-06 18:01:03|2026-05-06 18:03:05|
+-------------------+-------------------+-------------------+-------------------+



In [14]:
spark.sql("""
SELECT
  traffic_borough,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) AS aq_non_null_rows,
  ROUND(100.0 * SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS aq_non_null_pct
FROM local.db.enriched_traffic
GROUP BY traffic_borough
ORDER BY traffic_borough;
""").show(truncate=False)

+---------------+----------+----------------+---------------+
|traffic_borough|total_rows|aq_non_null_rows|aq_non_null_pct|
+---------------+----------+----------------+---------------+
|Bronx          |594       |330             |55.56          |
|Brooklyn       |321       |200             |62.31          |
|Manhattan      |1316      |1030            |78.27          |
|Queens         |677       |270             |39.88          |
|Staten Island  |406       |120             |29.56          |
+---------------+----------+----------------+---------------+

